In [63]:
import torch
import torch.nn as nn
import triton
import triton.language as tl
import time

In [64]:
def prepare_smoothquant_weights_per_layer(W, x_calib, clip_max=8.0):
    # W.shape = (out_features, in_features)
    act_scale = x_calib.abs().clamp(min=1e-2)  # shape = in_features
    W_scaled = W * act_scale[None, :]          # broadcast по in_features
    W_scaled = W_scaled.clamp(-clip_max, clip_max)
    W_max = W_scaled.abs().max(dim=0)[0]       # max по out_features
    W_q = ((W_scaled / W_max[None,:]*127).round().clamp(-127,127)).to(torch.int8)
    w_scale = W_max / 127
    return W_q, act_scale, w_scale

In [65]:
@triton.jit
def smoothquant_gemm(
    X_ptr, W_ptr, Y_ptr, S_ptr, W_scale_ptr,
    M, N, K,
    stride_xm, stride_xk,
    stride_wk, stride_wn,
    stride_ym, stride_yn,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, BLOCK_K)
    acc = tl.zeros((BLOCK_M, BLOCK_N), tl.int32)

    for k in range(0, K, BLOCK_K):
        x = tl.load(
            X_ptr + offs_m[:, None]*stride_xm + (k+offs_k)[None,:]*stride_xk,
            mask=(offs_m[:, None]<M) & (k+offs_k[None,:]<K),
            other=0.0
        )
        s = tl.load(S_ptr + k + offs_k)
        s = tl.where(s < 1e-6, 1e-6, s)
        x_scaled = x / s
        x_int8 = tl.where(x_scaled >= 0,
                          tl.floor(x_scaled + 0.5),
                          tl.ceil(x_scaled - 0.5)).to(tl.int8)
        w = tl.load(
            W_ptr + (k+offs_k)[:,None]*stride_wk + offs_n[None,:]*stride_wn,
            mask=(k+offs_k[:,None]<K) & (offs_n[None,:]<N),
            other=0
        )
        acc += tl.dot(x_int8, w)

    w_scale = tl.load(W_scale_ptr + offs_n)
    y = acc.to(tl.float32) * w_scale[None,:]
    tl.store(Y_ptr + offs_m[:, None]*stride_ym + offs_n[None,:]*stride_yn, y)

In [66]:
class SmoothQuantLinear(nn.Module):
    def __init__(self, W, x_calib):
        super().__init__()
        W_q, act_scale, w_scale = prepare_smoothquant_weights_per_layer(W, x_calib)
        self.register_buffer("weight", W_q)
        self.register_buffer("act_scale", act_scale)
        self.register_buffer("w_scale", w_scale)

    def forward(self, x):
        B, K = x.shape
        N = self.weight.shape[0]  # out_features
        y = torch.empty((B,N), device=x.device, dtype=torch.float16)
        grid = (triton.cdiv(B,128), triton.cdiv(N,128))
        smoothquant_gemm[grid](
            x, self.weight, y, self.act_scale, self.w_scale,
            B,N,K,
            x.stride(0), x.stride(1),
            self.weight.stride(0), self.weight.stride(1),
            y.stride(0), y.stride(1),
            BLOCK_M=128, BLOCK_N=128, BLOCK_K=32
        )
        return y

In [68]:
class MyMiniLM(nn.Module):
    def __init__(self, vocab_size=100, hidden=64, seq_len=8):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, hidden)
        self.fc1 = nn.Linear(hidden, hidden)
        self.fc2 = nn.Linear(hidden, vocab_size)

    def forward(self, x):
        h = self.embed(x)
        h = h.mean(dim=1)
        h = self.fc1(h)
        return self.fc2(h)

In [74]:
vocab_size = 100
seq_len = 180
batch = 64
x_data = torch.randint(0,vocab_size,(batch,seq_len)).cuda()
y_data = torch.randint(0,vocab_size,(batch,)).cuda()


model = ToyLM(vocab_size).cuda()
with torch.no_grad():
    hidden_embed = model.embed(x_data)               # shape [batch, seq_len, hidden]
x_calib_fc1 = hidden_embed.mean(dim=(0,1))          # shape [hidden]


with torch.no_grad():
    fc1_out = model.fc1(hidden_embed.mean(dim=1))   # shape [batch, hidden]
x_calib_fc2 = fc1_out.mean(dim=0)                   # shape [hidden]

model_sq = ToyLM(vocab_size).cuda()
model_sq.fc1 = SmoothQuantLinear(model.fc1.weight.data, x_calib_fc1)
model_sq.fc2 = SmoothQuantLinear(model.fc2.weight.data, x_calib_fc2)


def measure_latency(m, x):
    torch.cuda.synchronize()
    start = time.time()
    with torch.no_grad():
        _ = m(x)
    torch.cuda.synchronize()
    return time.time()-start

lat_orig = measure_latency(model, x_data)
lat_sq   = measure_latency(model_sq, x_data)
print("Latency original:", lat_orig)
print("Latency SmoothQuant:", lat_sq)


criterion = nn.CrossEntropyLoss()
loss_orig = criterion(model(x_data), y_data)
loss_sq   = criterion(model_sq(x_data), y_data)
print("Loss original:", loss_orig.item())
print("Loss SmoothQuant:", loss_sq.item())


Latency original: 0.000270843505859375
Latency SmoothQuant: 0.0005924701690673828
Loss original: 4.599663257598877
Loss SmoothQuant: 4.60546875
